In [46]:
import numpy as np
import pandas as pd
import time
from nltk.stem import PorterStemmer
from sklearn.model_selection import train_test_split

# load dataset buku
df = pd.read_csv('fix.csv')
print(df.head())
data = df[['title','authors','categories','description']]
print(data.head())
data.isnull().sum()
data = data.fillna('')
print(data.isnull().sum())

          isbn13      isbn10           title subtitle  \
0  9780002005883  0002005883          Gilead      NaN   
1  9780002261982  0002261987    Spider's Web  A Novel   
2  9780006163831  0006163831    The One Tree      NaN   
3  9780006178736  0006178731  Rage of angels      NaN   
4  9780006280897  0006280897  The Four Loves      NaN   

                           authors                     categories  \
0               Marilynne Robinson                        Fiction   
1  Charles Osborne;Agatha Christie  Detective and mystery stories   
2             Stephen R. Donaldson               American fiction   
3                   Sidney Sheldon                        Fiction   
4              Clive Staples Lewis                 Christian life   

                                           thumbnail  \
0  http://books.google.com/books/content?id=KQZCP...   
1  http://books.google.com/books/content?id=gA5GP...   
2  http://books.google.com/books/content?id=OmQaw...   
3  http://books.go

In [47]:
data['combined'] = data['title'] + " " + data['authors'] + " " + data['categories'] + " " + data['description']
print(data['combined'].iloc[0])
data['combined'] = data['combined'].str.lower()
print(data['combined'].iloc[0])
data['tokens'] = data['combined'].apply(lambda x: x.split())
print(data['tokens'].iloc[0])
stopwords = ['and','a','about','the','of','is','that']
data['filtered'] = data['tokens'].apply(lambda x: [w for w in x if w not in stopwords])
print(data['filtered'].iloc[0])
stemmer = PorterStemmer()
data['stemmed'] = data['filtered'].apply(
    lambda x: [stemmer.stem(word) for word in x]
)
print(data['stemmed'].iloc[0])
data['final'] = data['stemmed'].apply(lambda x: ' '.join(x))
print(data['final'].iloc[0])

Gilead Marilynne Robinson Fiction A NOVEL THAT READERS and critics have been eagerly anticipating for over a decade, Gilead is an astonishingly imagined story of remarkable lives. John Ames is a preacher, the son of a preacher and the grandson (both maternal and paternal) of preachers. It’s 1956 in Gilead, Iowa, towards the end of the Reverend Ames’s life, and he is absorbed in recording his family’s story, a legacy for the young son he will never see grow up. Haunted by his grandfather’s presence, John tells of the rift between his grandfather and his father: the elder, an angry visionary who fought for the abolitionist cause, and his son, an ardent pacifist. He is troubled, too, by his prodigal namesake, Jack (John Ames) Boughton, his best friend’s lost son who returns to Gilead searching for forgiveness and redemption. Told in John Ames’s joyous, rambling voice that finds beauty, humour and truth in the smallest of life’s details, Gilead is a song of celebration and acceptance of th

In [48]:
train_data, test_data = train_test_split(data, test_size=0.2, random_state=42)

train_data = train_data.reset_index(drop=True)
test_data = test_data.reset_index(drop=True)

print("Jumlah Data Train :", len(train_data))
print("Jumlah Data Test  :", len(test_data))

Jumlah Data Train : 5448
Jumlah Data Test  : 1362


In [49]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Hyperparameter tuning
tfidf = TfidfVectorizer(
    stop_words='english',
    max_features=5000,
    ngram_range=(1,2),
    min_df=2,
    max_df=0.8
)

# TF-IDF matrix
tfidf_train = tfidf.fit_transform(train_data['final'])
tfidf_test = tfidf.transform(test_data['final'])

# Cosine similarity
cosine_sim = cosine_similarity(tfidf_test, tfidf_train)

# cek hasil
print(tfidf_train.shape)
print(cosine_sim[:5])

(5448, 5000)
[[0.         0.09511077 0.00786168 ... 0.         0.01311304 0.02502198]
 [0.         0.08198789 0.0155174  ... 0.         0.00605277 0.04347104]
 [0.         0.00243479 0.03483796 ... 0.08694186 0.01479824 0.00420338]
 [0.01761426 0.         0.         ... 0.         0.02810694 0.02683136]
 [0.         0.03238346 0.07341889 ... 0.         0.01648973 0.01618539]]


In [50]:
def rekomendasi(judul):
    judul = str(judul).strip().lower()
    
    test_data['title_clean'] = test_data['title'].astype(str).str.lower()
    hasil = test_data[test_data['title_clean'].str.contains(judul, na=False)]
    
    if hasil.empty:
        print("Judul tidak ditemukan di Data Uji")
        return
    
    idx = hasil.index[0]
    
    skor = list(enumerate(cosine_sim[idx]))
    skor = sorted(skor, key=lambda x: x[1], reverse=True)
    
    print("Rekomendasi untuk:", test_data.loc[idx, 'title'])
    print("-" * 30)
    
    count = 0
    for i in skor:
        print(train_data.loc[i[0], 'title'], "->", round(i[1], 3))
        count += 1
        if count == 3:
            break

rekomendasi("harry potter")

Rekomendasi untuk: Harry Potter and the Sorcerer's Stone (Book 1)
------------------------------
Harry Potter and the Chamber of Secrets (Book 2) -> 0.55
Harry Potter and the Half-Blood Prince (Book 6) -> 0.545
Harry Potter and the Order of the Phoenix (Book 5) -> 0.522


In [51]:
def precision_at_k(idx, cosine_sim, train_df, test_df, k=3):
    skor = list(enumerate(cosine_sim[idx]))
    skor = sorted(skor, key=lambda x: x[1], reverse=True)

    rekom = [i[0] for i in skor[:k]]
    kategori_asli = set(str(test_df.loc[idx, 'categories']).lower().split())

    relevan = 0
    for i in rekom:
        kategori_rekom = set(str(train_df.loc[i, 'categories']).lower().split())
        if len(kategori_asli & kategori_rekom) > 0:
            relevan += 1

    return relevan / k

def ndcg_at_k(idx, cosine_sim, train_df, test_df, k=3):
    skor = list(enumerate(cosine_sim[idx]))
    skor = sorted(skor, key=lambda x: x[1], reverse=True)

    rekom = [i[0] for i in skor[:k]]
    kategori_asli = set(str(test_df.loc[idx, 'categories']).lower().split())

    relevansi = []
    for i in rekom:
        kategori_rekom = set(str(train_df.loc[i, 'categories']).lower().split())
        rel = 1 if len(kategori_asli & kategori_rekom) > 0 else 0
        relevansi.append(rel)

    # DCG
    dcg_val = sum([rel / np.log2(i + 2) for i, rel in enumerate(relevansi)])

    # IDCG
    ideal = sorted(relevansi, reverse=True)
    idcg_val = sum([rel / np.log2(i + 2) for i, rel in enumerate(ideal)])

    if idcg_val == 0:
        return 0

    return dcg_val / idcg_val

In [52]:
# Fungsi baru ini menggabungkan Precision dan NDCG agar tidak perlu sorting 2 kali
def hitung_metrik(idx, cosine_sim_row, train_df, test_df, k=3):
    # Menggunakan np.argsort dari Numpy agar sorting berkali-kali lipat lebih cepat
    # Mengambil k rekomendasi teratas
    rekom_idx = np.argsort(cosine_sim_row)[-k:][::-1]

    kategori_asli = set(str(test_df.loc[idx, 'categories']).lower().split())

    relevansi = []
    relevan_count = 0

    for i in rekom_idx:
        kategori_rekom = set(str(train_df.loc[i, 'categories']).lower().split())
        if len(kategori_asli & kategori_rekom) > 0:
            relevansi.append(1)
            relevan_count += 1
        else:
            relevansi.append(0)

    # 1. Hitung Precision@K
    precision_val = relevan_count / k

    # 2. Hitung NDCG@K
    dcg_val = sum([rel / np.log2(i + 2) for i, rel in enumerate(relevansi)])
    ideal = sorted(relevansi, reverse=True)
    idcg_val = sum([rel / np.log2(i + 2) for i, rel in enumerate(ideal)])

    ndcg_val = 0 if idcg_val == 0 else dcg_val / idcg_val

    return precision_val, ndcg_val


# Fungsi evaluasi yang sudah dioptimasi
def evaluasi_sistem(train_df, test_df, cosine_sim, k=3):
    start_time = time.time()   # mulai hitung waktu

    total_precision = 0
    total_ndcg = 0

    for idx in range(len(test_df)):
        # Panggil satu fungsi hitung_metrik saja, dapat prec dan ndcg sekaligus!
        prec, ndcg = hitung_metrik(idx, cosine_sim[idx], train_df, test_df, k)
        total_precision += prec
        total_ndcg += ndcg

    avg_precision = total_precision / len(test_df)
    avg_ndcg = total_ndcg / len(test_df)

    end_time = time.time()   # selesai hitung waktu
    running_time = end_time - start_time

    # Output tetap persis seperti yang Anda buat sebelumnya
    print("Precision@{} :".format(k), round(avg_precision, 3))
    print("NDCG@{}      :".format(k), round(avg_ndcg, 3))
    print("Running Time :", round(running_time, 4), "detik")
    print("-" * 30) # tambahan garis pembatas agar output k=3, 5, 10 lebih mudah dibaca

    return avg_precision, avg_ndcg, running_time


# === PEMANGGILAN FUNGSI (TETAP SAMA) ===
rekomendasi("harry potter")

print("\n--- HASIL EVALUASI ---")
evaluasi_sistem(train_data, test_data, cosine_sim, k=3)
evaluasi_sistem(train_data, test_data, cosine_sim, k=5)
evaluasi_sistem(train_data, test_data, cosine_sim, k=10)

Rekomendasi untuk: Harry Potter and the Sorcerer's Stone (Book 1)
------------------------------
Harry Potter and the Chamber of Secrets (Book 2) -> 0.55
Harry Potter and the Half-Blood Prince (Book 6) -> 0.545
Harry Potter and the Order of the Phoenix (Book 5) -> 0.522

--- HASIL EVALUASI ---
Precision@3 : 0.523
NDCG@3      : 0.666
Running Time : 0.3051 detik
------------------------------
Precision@5 : 0.518
NDCG@5      : 0.688
Running Time : 0.3117 detik
------------------------------
Precision@10 : 0.508
NDCG@10      : 0.706
Running Time : 0.3469 detik
------------------------------


(0.508296622613804, np.float64(0.7059711054932061), 0.3468921184539795)